# 09.04 · Model Context Protocol (MCP)

**Chapter:** Tool Use & Agents · **Milestone:** M4

## Goals
- [ ] MCP's architecture: hosts, clients, servers; tools, resources, prompts; stdio vs HTTP transports
- [ ] Build a server that exposes your RAG index (`mcp_server.py` next to this notebook)
- [ ] Connect to it from Python, from an LLM agent, and from Claude Code / Claude Desktop / IDEs

## Resources
- [MCP docs](https://modelcontextprotocol.io/) · [Python SDK](https://github.com/modelcontextprotocol/python-sdk) — free
- [Hugging Face MCP course](https://huggingface.co/learn/mcp-course) — free
- [MCP servers directory](https://github.com/modelcontextprotocol/servers) — free

**Why MCP**: before, every app wrote its own integration for every tool (N apps × M tools). MCP is a standard plug: write a server once,
use it from any MCP host (Claude Desktop/Code, ChatGPT, Cursor, VS Code, Zed, your own agent). Think "USB-C for tools and context".

| Concept | What | Example |
|---|---|---|
| **Host** | the app the user talks to | Claude Code, an IDE, your agent |
| **Client** | connection from host to one server | `ClientSession` below |
| **Server** | exposes capabilities | `mcp_server.py` |
| **Tools** | functions the model can call | `search_docs`, `get_passage` |
| **Resources** | read-only data the host can load into context | `wiki://stats` |
| **Prompts** | reusable prompt templates the user can pick | `cite_answer` |

In [ ]:
import json
import sys
from pathlib import Path

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

from aieng.llm import client

print(Path("mcp_server.py").read_text())

## 1. Talk to the server as a client
The client launches the server as a subprocess and speaks JSON-RPC over stdin/stdout.

In [ ]:
SERVER = StdioServerParameters(command=sys.executable, args=["mcp_server.py"])

async with stdio_client(SERVER) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = (await session.list_tools()).tools
        for t in tools:
            print(f"tool {t.name}: {t.description}\n  schema: {json.dumps(t.input_schema)}")
        print("resources:", [r.uri for r in (await session.list_resources()).resources])
        print("prompts:", [p.name for p in (await session.list_prompts()).prompts])
        print("\nstats:", (await session.read_resource("wiki://stats")).contents[0].text)
        result = await session.call_tool("search_docs", {"query": "Lincoln's wife", "k": 2})
        print("\nsearch_docs ->", result.content[0].text[:300])

## 2. An LLM agent that uses MCP tools
Bridge: MCP tool list → OpenAI tool schemas; model's tool calls → `session.call_tool`. Any MCP server now plugs into your 09.02 loop.

In [ ]:
c, MODEL = client()
EXTRA = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}


def to_openai_tool(t):
    return {"type": "function", "function": {"name": t.name, "description": t.description or "", "parameters": t.input_schema}}


async def mcp_agent(question, max_steps=5):
    async with stdio_client(SERVER) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = [to_openai_tool(t) for t in (await session.list_tools()).tools]
            messages = [{"role": "system", "content": "Answer using the tools. Cite passage ids like [12]."},
                        {"role": "user", "content": question}]
            for step in range(max_steps):
                r = c.chat.completions.create(model=MODEL, messages=messages, tools=tools, temperature=0, max_tokens=800,
                                              tool_choice="required" if step == 0 else "auto", **EXTRA)
                msg = r.choices[0].message
                if not msg.tool_calls and step == 0:  # Ollama ignores tool_choice: nudge and retry instead
                    messages.append({"role": "user", "content": "Use the tools before answering."})
                    continue
                if not msg.tool_calls:
                    return msg.content
                messages.append(msg.model_dump(exclude_none=True))
                for call in msg.tool_calls:
                    res = await session.call_tool(call.function.name, json.loads(call.function.arguments or "{}"))
                    text = "\n".join(getattr(part, "text", "") for part in res.content)
                    print(f"  [mcp] {call.function.name}({call.function.arguments}) -> {len(text)} chars")
                    messages.append({"role": "tool", "tool_call_id": call.id, "content": text})
            return "step limit reached"


print(await mcp_agent("When did Lincoln marry Mary Todd?"))

## 3. Use it from real MCP hosts
**Claude Code** (from the repo root):
```bash
claude mcp add mini-wiki -- uv run --directory "$(pwd)" python chapters/09-agents/mcp_server.py
claude   # then ask: "use mini-wiki to find when Lincoln was born"
```
**Claude Desktop / Cursor / VS Code** — add to their MCP config JSON:
```json
{"mcpServers": {"mini-wiki": {"command": "uv",
  "args": ["run", "--directory", "/ABSOLUTE/PATH/ai-eng", "python", "chapters/09-agents/mcp_server.py"]}}}
```
**MCP Inspector** (debug UI): `npx @modelcontextprotocol/inspector uv run python chapters/09-agents/mcp_server.py`

## 4. Transports and security
- **stdio**: local subprocess; simplest; the server runs with *your* permissions.
- **Streamable HTTP**: remote servers (`mcp.run(transport="streamable-http")`), with OAuth for auth.
- Installing an MCP server = running someone's code with your permissions, and its tool descriptions/results go straight into the model's
  context (prompt-injection vector, "tool poisoning"). Review servers like dependencies; prefer least-privilege tools.

## Exercises
1. Add a `stats` **tool** that returns corpus statistics, and a resource template `wiki://passage/{id}`. Test them with the Inspector.
2. Connect the server to Claude Code (above) and ask a multi-part question. Watch which tools it calls.
3. Write a second MCP server exposing the `shop.sqlite` text-to-SQL tool from 08.05 (read-only!). Use both servers from `mcp_agent`.

In [ ]:
# your code here